# 3 - RAG Evaluation (LLM as a Judge)

Retrieval metrics don't tell us whether the final *answers* are good. Here we
evaluate the full RAG pipeline with **LLM-as-a-Judge** (module 4) and compare
**two prompt variants**, plus a **query-rewriting** experiment (module 6 /
best practices).

1. Sample 100 ground-truth questions
2. Generate answers with two different system prompts (same retrieval)
3. Judge every answer: RELEVANT / PARTLY_RELEVANT / NON_RELEVANT
4. Compare the distributions and pick the better prompt
5. Bonus: does LLM query rewriting improve *retrieval*?

Requires the Docker stack running. Cost: a few hundred small LLM calls.

In [1]:
import json
import os
import random

import pandas as pd
import psycopg
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from tqdm.auto import tqdm

load_dotenv("../.env")

client = OpenAI()
MODEL = "gpt-5.4-mini"
EMBED_MODEL = "text-embedding-3-small"

conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    autocommit=True,
)

In [2]:
df_gt = pd.read_csv("../data/ground-truth.csv")

N_SAMPLE = 100
df_sample = df_gt.sample(N_SAMPLE, random_state=1).reset_index(drop=True)
len(df_sample)

100

## RAG pipeline (mirrors the n8n flow)

Same knowledge base, same embedding model, same top-k as the live n8n
workflow - only the prompt varies. We re-implement it in Python because
batch experiments over 100+ questions are much easier here than in n8n
(that's also why the course does offline evaluation in notebooks).

In [3]:
def embed(text):
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return response.data[0].embedding


def vector_search(query_vector, k=5):
    return conn.execute(
        '''
        SELECT text, metadata->>'url'
        FROM n8n_vectors
        ORDER BY embedding <=> %s::vector
        LIMIT %s
        ''',
        (str(query_vector), k),
    ).fetchall()


def build_context(results):
    return "\n\n---\n\n".join(f"{text}\n(source: {url})" for text, url in results)


def llm(system_prompt, user_prompt):
    response = client.responses.create(
        model=MODEL,
        input=[
            {"role": "developer", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.output_text


def rag(question, system_prompt, k=5):
    results = vector_search(embed(question), k=k)
    user_prompt = f"QUESTION: {question}\n\nCONTEXT:\n{build_context(results)}"
    return llm(system_prompt, user_prompt)

## The two prompt variants

- **strict**: course-style minimal prompt - answer only from context
- **guided**: the production system prompt from the n8n workflow - persona,
  step-by-step instructions, sources

In [4]:
PROMPT_STRICT = '''
Answer the QUESTION based only on the CONTEXT from the n8n documentation.
If the context does not contain the answer, say "I don't know".
'''.strip()

PROMPT_GUIDED = '''
You are the n8n platform support assistant for our company. Employees ask
you how to build and run workflows in n8n.

Answer the QUESTION based only on the CONTEXT from the official n8n
documentation. If the context does not contain the answer, say that you
don't know and recommend asking in the #platform-support channel. Never
invent nodes, parameters, or features.

Keep answers practical and concise. When it helps, give step-by-step
instructions. End with a 'Sources:' line listing the source urls you used.
'''.strip()

PROMPTS = {"strict": PROMPT_STRICT, "guided": PROMPT_GUIDED}

In [5]:
rag(df_sample.question[0], PROMPT_GUIDED)

'Here’s how those settings affect node execution in n8n:\n\n- **Execute Once**: the node runs only once, using data from the **first incoming item**. It does **not** process any extra items.\n- **Retry On Fail**: if the node execution fails, n8n will **rerun it until it succeeds**.\n- **On Error**:\n  - **Stop Workflow**: stops the whole workflow immediately, so no later nodes run.\n  - **Continue**: the workflow keeps going to the next node, using the **last valid data**.\n  - **Continue (using error output)**: the workflow keeps going and passes the **error information** to the next node, so you can handle it there.\n\nIf you want a workflow to fail on purpose and trigger an error workflow, you can use the **Stop And Error** node.\n\nSources:\n- https://docs.n8n.io/build/understand-workflows/workflow-components/work-with-nodes/\n- https://docs.n8n.io/build/flow-logic/handle-errors-gracefully/'

In [6]:
answers = {}

for name, prompt in PROMPTS.items():
    answers[name] = [
        rag(q, prompt) for q in tqdm(df_sample.question, desc=name)
    ]

strict:   0%|          | 0/100 [00:00<?, ?it/s]

guided:   0%|          | 0/100 [00:00<?, ?it/s]

## LLM as a Judge

We use the *question + answer* variant of the judge prompt (module 4): the
judge doesn't see the context, it only rates whether the answer actually
addresses the question.

In [7]:
class Verdict(BaseModel):
    verdict: str
    explanation: str


JUDGE_PROMPT = '''
You are an expert evaluator for a RAG system that answers questions about
the n8n documentation.

Classify how relevant the generated answer is to the question:
- "RELEVANT": directly and completely addresses the question
- "PARTLY_RELEVANT": addresses it only partially or vaguely
- "NON_RELEVANT": does not address the question, or only says "I don't know"

Question: {question}

Answer: {answer}
'''.strip()


def judge(question, answer):
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": JUDGE_PROMPT.format(question=question, answer=answer)}],
        text_format=Verdict,
    )
    return response.output_parsed.verdict

In [8]:
verdicts = {}

for name in PROMPTS:
    verdicts[name] = [
        judge(q, a)
        for q, a in tqdm(zip(df_sample.question, answers[name]), total=N_SAMPLE, desc=name)
    ]

strict:   0%|          | 0/100 [00:00<?, ?it/s]

guided:   0%|          | 0/100 [00:00<?, ?it/s]

In [9]:
df_verdicts = pd.DataFrame({
    name: pd.Series(v).value_counts(normalize=True)
    for name, v in verdicts.items()
}).fillna(0)
df_verdicts.round(3)

,strict,guided
NON_RELEVANT,0.01,0.00
PARTLY_RELEVANT,0.06,0.03
RELEVANT,0.93,0.97


The prompt with the higher RELEVANT share wins - that's the one configured
in the n8n workflow's system message.

## Bonus: query rewriting (best practice)

Users often type terse queries ("loop items??"). We let an LLM rewrite the
question into a precise search query first, and measure whether *retrieval*
improves. (We evaluate on retrieval hit rate, because that's what rewriting
directly affects.)

In [10]:
REWRITE_PROMPT = '''
Rewrite the user question as a precise, self-contained search query for the
official n8n documentation. Expand abbreviations, name the n8n concepts
involved, and keep it to one sentence. Return only the rewritten query.

Question: {question}
'''.strip()


def rewrite(question):
    return llm(
        "You rewrite search queries for a documentation search engine.",
        REWRITE_PROMPT.format(question=question),
    )

rewrite("how loop over stuff")

'n8n loop through items using the Loop Over Items node and iterating over data in a workflow'

In [11]:
def doc_hit(question_text, expected_doc_id, k=5):
    results = conn.execute(
        '''
        SELECT metadata->>'doc_id'
        FROM n8n_vectors
        ORDER BY embedding <=> %s::vector
        LIMIT %s
        ''',
        (str(embed(question_text)), k),
    ).fetchall()
    return any(row[0] == expected_doc_id for row in results)


hits_original, hits_rewritten = [], []

for record in tqdm(df_sample.to_dict(orient="records")):
    hits_original.append(doc_hit(record["question"], record["doc_id"]))
    hits_rewritten.append(doc_hit(rewrite(record["question"]), record["doc_id"]))

pd.DataFrame({
    "approach": ["original question", "LLM-rewritten query"],
    "hit_rate_doc@5": [
        sum(hits_original) / len(hits_original),
        sum(hits_rewritten) / len(hits_rewritten),
    ],
}).round(3)

  0%|          | 0/100 [00:00<?, ?it/s]

,approach,hit_rate_doc@5
0,original question,0.95
1,LLM-rewritten query,0.87


## Conclusion

- **Prompt comparison:** the *guided* production prompt clearly wins - 97%
  RELEVANT (0% non-relevant) vs 93% for the minimal *strict* prompt. The
  guided prompt is what the production n8n workflow uses in its system
  message.
- **Query rewriting:** an honest negative result - rewriting *lowered* the
  doc-level hit rate from 0.95 to 0.87 on this sample. Ground-truth
  questions are already precise, so the rewrite step only adds drift.
  Consequently the live flow does **not** rewrite queries as a fixed step;
  the n8n agent still formulates its own tool queries dynamically, which
  covers the messy-real-query case rewriting is meant for.
- The **live monitoring judge** (in the n8n workflow) applies the same
  LLM-as-a-judge idea to real traffic - see the Grafana dashboard.
